# Egyptian plate OCR: local review workflow

This replaces the platform-specific experiment with a portable entry point to the reviewed package. It deliberately changes the old workflow: no package-install cells, Roboflow, Kaggle paths, automatic checkpoint downloads, GPU assumption, implicit training, export or upload. Install the locked inference environment before opening this notebook. Run from the repository root.

The decoder example below is synthetic, not image recognition. Training and real prediction are opt-in and require trusted local artifacts with known permissions. Historical detection metrics are not rerun or plate-recognition accuracy. Original exploratory visualization cells are no longer part of this runnable workflow; Git history is unchanged.


In [ ]:
import json
import os
from pathlib import Path

from egypt_plate_ocr.core import Detection, decode_characters
from egypt_plate_ocr.inference import predict_plates
from egypt_plate_ocr.training import train_local

print(json.dumps({"mode": "local-only", "training_enabled": False}))


## Decoder wiring only
No image, weights, credentials or network is needed. Invented detections test the package ordering, not a trained detector.


In [ ]:
decoder_result = decode_characters([Detection("baa", 10), Detection("1", 20)])
assert decoder_result == {"letters": "ب", "numbers": "١"}
print(json.dumps({"synthetic": True, "decoder": decoder_result}, ensure_ascii=False))


## Optional CPU training
Leave `OCR_RUN_TRAINING` unset for the safe default. To train deliberately, set it to `1` and provide `OCR_DATA_YAML`, `OCR_BASE_WEIGHTS` and a new `OCR_TRAIN_OUTPUT`. The YAML must point only to permitted local images and labels, with independent train/validation splits. The wrapper rejects download directives, remote split URLs, missing directories and identical train/validation directories. Review local dataset provenance and image-level overlap yourself. Use a network-disabled environment to enforce offline operation. Local `.pt` files can execute code when loaded, so never use untrusted checkpoints. The one-epoch default is a smoke setting, not a completed model.


In [ ]:
if os.environ.get("OCR_RUN_TRAINING") == "1":
    required = ("OCR_DATA_YAML", "OCR_BASE_WEIGHTS", "OCR_TRAIN_OUTPUT")
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise ValueError("Missing explicit training settings: " + ", ".join(missing))
    training_result = train_local(
        Path(os.environ["OCR_DATA_YAML"]),
        Path(os.environ["OCR_BASE_WEIGHTS"]),
        Path(os.environ["OCR_TRAIN_OUTPUT"]),
        epochs=int(os.environ.get("OCR_EPOCHS", "1")),
    )
    print(json.dumps(training_result))
else:
    print("Training skipped. No model or dataset is loaded.")


## Optional image inference
Set both `OCR_IMAGE` and `OCR_WEIGHTS` to existing trusted local files. If either is supplied, both are required. Results are detector outputs, not validated registration numbers. There is no automatic reuse of an image or training artifact from another cell.


In [ ]:
image = os.environ.get("OCR_IMAGE")
weights = os.environ.get("OCR_WEIGHTS")
if image or weights:
    if not image or not weights:
        raise ValueError("Supply both OCR_IMAGE and OCR_WEIGHTS.")
    print(json.dumps(predict_plates(Path(image), Path(weights)), ensure_ascii=False))
else:
    print("Image inference skipped. Supply explicit trusted local artifacts to enable it.")
